In [ ]:
#Import Required Libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
#Load and Access the Dataset (MNIST)
(x_train, _), (x_test, _) = keras.datasets.mnist.load_data()

# Normalize and reshape
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
x_train = x_train.reshape((len(x_train), 28, 28, 1))
x_test = x_test.reshape((len(x_test), 28, 28, 1))


In [ ]:
#Build the Autoencoder Model
# --- Encoder ---
encoder = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(16, (3,3), activation='relu', padding='same'),
    layers.MaxPooling2D((2,2), padding='same'),
    layers.Conv2D(8, (3,3), activation='relu', padding='same'),
    layers.MaxPooling2D((2,2), padding='same')
])

# --- Decoder ---
decoder = keras.Sequential([
    layers.Conv2D(8, (3,3), activation='relu', padding='same', input_shape=(7,7,8)),
    layers.UpSampling2D((2,2)),
    layers.Conv2D(16, (3,3), activation='relu', padding='same'),
    layers.UpSampling2D((2,2)),
    layers.Conv2D(1, (3,3), activation='sigmoid', padding='same')
])

# --- Combine Encoder + Decoder ---
autoencoder = keras.Sequential([encoder, decoder])


In [ ]:
#Compile the Model (Optimizer, Loss, Metrics)
autoencoder.compile(optimizer='adam', loss='mse', metrics=['accuracy'])


In [ ]:
#Train the Model
history = autoencoder.fit(x_train, x_train,
                          epochs=5,
                          batch_size=128,
                          validation_split=0.1,
                          verbose=1)


In [ ]:
#Evaluate and Visualize Results
decoded_imgs = autoencoder.predict(x_test[:10])

plt.figure(figsize=(10, 4))
for i in range(10):
    # Original
    plt.subplot(2, 10, i + 1)
    plt.imshow(x_test[i].reshape(28, 28), cmap="gray")
    plt.axis("off")

    # Reconstructed
    plt.subplot(2, 10, i + 11)
    plt.imshow(decoded_imgs[i].reshape(28, 28), cmap="gray")
    plt.axis("off")

plt.suptitle("Top: Original Images | Bottom: Reconstructed (Autoencoder Output)")
plt.show()

In [ ]:
#Anomaly Detection (High Reconstruction Error)
reconstructions = autoencoder.predict(x_test)
mse = np.mean(np.power(x_test - reconstructions, 2), axis=(1,2,3))
threshold = np.mean(mse) + 2*np.std(mse)

print(f"Anomaly Detection Threshold: {threshold:.6f}")

# Example: Detect anomalies
anomalies = mse > threshold
print(f"Detected {np.sum(anomalies)} anomalies out of {len(x_test)} samples.")